## Configurable Decorators: Decorators with Arguments

- A basic decorator adds fixed behavior; sometimes you need to *configure* that behaviour (e.g. how many retries, which log level).  
- You cannot pass options directly to a plain `@decorator`, because that decorator receives only the target function.  
- Solution: call a *factory* that takes options and returns a decorator, then apply it with `@factory(option=value)`.  

## The Decorator Factory Pattern

- **Factory function** receives configuration arguments and returns the **actual decorator**.  
- The actual **decorator** still takes the target function and builds a **wrapper**.  
- The wrapper can access both the factory’s configuration (via a closure) and the call‑time `*args / **kwargs` for the target function.  
- Three nested layers keep concerns separated: configuration ➜ decoration ➜ runtime.

### Applying Decorators with Arguments

- Use `@factory(arg1, arg2…)` above the function definition.  
- At *definition* time Python calls the factory, gets back a decorator, and applies that decorator to the function.  
- Callers of the function automatically get the behaviour configured by the factory.

## Example: Retry Decorator Factory

- A practical DevOps scenario: retry a flaky operation a configurable number of times.  
- The factory takes `max_attempts`; the wrapper loops until success or until attempts are exhausted, re‑raising the last error.  

In [14]:
import random

def retry(func):
    max_attempt = 5
    def wrapper(*args, **kwargs):
        for attempt in range(1,max_attempt+1):
            try:
                print(f"Attempt {attempt}/{max_attempt}")
                return func(*args, **kwargs)
            except Exception as e:
                print(f" Error: {e}")
                if attempt == max_attempt:
                    raise
    return wrapper
    
def sometimes_fails():
    if random.random() < 0.7:
        raise RuntimeError("Flaky failure")
    return "Success!"

sometimes_fails = retry(sometimes_fails)
print(f"Result: {sometimes_fails()}")

Attempt 1/5
 Error: Flaky failure
Attempt 2/5
 Error: Flaky failure
Attempt 3/5
 Error: Flaky failure
Attempt 4/5
 Error: Flaky failure
Attempt 5/5
Result: Success!


In [15]:
import random

def retry(func):
    max_attempt = 5
    def wrapper(*args, **kwargs):
        for attempt in range(1,max_attempt+1):
            try:
                print(f"Attempt {attempt}/{max_attempt}")
                return func(*args, **kwargs)
            except Exception as e:
                print(f" Error: {e}")
                if attempt == max_attempt:
                    raise
    return wrapper

@retry
def sometimes_fails():
    if random.random() < 0.7:
        raise RuntimeError("Flaky failure")
    return "Success!"

#sometimes_fails = retry(sometimes_fails)
print(f"Result: {sometimes_fails()}")

Attempt 1/5
Result: Success!


In [18]:
import random

def retry(max_attempt=5):
    def decorater(func):
        #max_attempt = 5
        def wrapper(*args, **kwargs):
            for attempt in range(1,max_attempt+1):
                try:
                    print(f"Attempt {attempt}/{max_attempt}")
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f" Error: {e}")
                    if attempt == max_attempt:
                        raise
        return wrapper
    return decorater

@retry(5)
def sometimes_fails():
    if random.random() < 0.7:
        raise RuntimeError("Flaky failure")
    return "Success!"

#sometimes_fails = retry(sometimes_fails)
print(f"Result: {sometimes_fails()}")

Attempt 1/5
Result: Success!
